In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

In [ ]:
FINAL_DATA_BASE = Path("/Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data")

# Lokaler Dokumentations-/Output-Ordner
LOCAL_OUT_BASE = FINAL_DATA_BASE / "FoldVision"
LOCAL_OUT_BASE.mkdir(parents=True, exist_ok=True)

# HPC paths, die später im Job-Skript verwendet werden
HPC_PROJECT_BASE = Path("/gpfs/project/projects/CompCellBio/PETaseTournament")
HPC_FOLDVISION_BASE = HPC_PROJECT_BASE / "FoldVision"
HPC_FOLDVISION_DATA_BASE = HPC_FOLDVISION_BASE / "data"
HPC_MUT_PDB_BASE = HPC_PROJECT_BASE / "mut_pdbs"

print("Final data base:", FINAL_DATA_BASE)
print("Local output base:", LOCAL_OUT_BASE)
print("HPC FoldVision data base:", HPC_FOLDVISION_DATA_BASE)
print("HPC mutated PDB base:", HPC_MUT_PDB_BASE)

Final data base: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data
Local output base: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/FoldVision
HPC FoldVision data base: /gpfs/project/projects/CompCellBio/PETaseTournament/FoldVision/data
HPC mutated PDB base: /gpfs/project/projects/CompCellBio/PETaseTournament/mut_pdbs


In [ ]:
DATASETS = [
    {
        "dataset": "UBE4B",
        "split_dir": FINAL_DATA_BASE / "UBE4B",
        "hpc_pdb_dir": HPC_MUT_PDB_BASE / "UBE4B",
        "hpc_fv_name": "UBE4B",
        "pdb_prefix": "UBE4B",
    },
    {
        "dataset": "GRB2",
        "split_dir": FINAL_DATA_BASE / "GRB2",
        "hpc_pdb_dir": HPC_MUT_PDB_BASE / "GRB2",
        "hpc_fv_name": "GRB2",
        "pdb_prefix": "GRB2",
    },
    {
        "dataset": "PTEN_activity",
        "split_dir": FINAL_DATA_BASE / "PTEN_activity",
        "hpc_pdb_dir": HPC_MUT_PDB_BASE / "PTEN_activity",
        "hpc_fv_name": "PTEN_activity",
        "pdb_prefix": "PTEN",
    },
    {
        "dataset": "PTEN_abundance",
        "split_dir": FINAL_DATA_BASE / "PTEN_abundance",
        "hpc_pdb_dir": HPC_MUT_PDB_BASE / "PTEN_abundance",
        "hpc_fv_name": "PTEN_abundance",
        "pdb_prefix": "PTEN",
    },
    {
        "dataset": "Alpha-Amylase_expression",
        "split_dir": FINAL_DATA_BASE / "Alpha_Amylase_expression",
        "hpc_pdb_dir": HPC_MUT_PDB_BASE / "Alpha_Amylase" / "expression",
        "hpc_fv_name": "Alpha-Amylase_expression",
        "pdb_prefix": "Alpha_Amylase",
    },
    {
        "dataset": "Alpha-Amylase_thermostability",
        "split_dir": FINAL_DATA_BASE / "Alpha_Amylase_thermostability",
        "hpc_pdb_dir": HPC_MUT_PDB_BASE / "Alpha_Amylase" / "thermostability",
        "hpc_fv_name": "Alpha-Amylase_thermostability",
        "pdb_prefix": "Alpha_Amylase",
    },
    {
        "dataset": "Alpha-Amylase_specific_activity",
        "split_dir": FINAL_DATA_BASE / "Alpha_Amylase_activity",
        "hpc_pdb_dir": HPC_MUT_PDB_BASE / "Alpha_Amylase" / "activity",
        "hpc_fv_name": "Alpha-Amylase_specific_activity",
        "pdb_prefix": "Alpha_Amylase",
    },
    {
        "dataset": "DLG4_abundance",
        "split_dir": FINAL_DATA_BASE / "DLG4_abundance",
        "hpc_pdb_dir": HPC_MUT_PDB_BASE / "DLG4_abundance",
        "hpc_fv_name": "DLG4_abundance",
        "pdb_prefix": "DLG4",
    },
    {
        "dataset": "DLG4_binding",
        "split_dir": FINAL_DATA_BASE / "DLG4_binding",
        "hpc_pdb_dir": HPC_MUT_PDB_BASE / "DLG4_binding",
        "hpc_fv_name": "DLG4_binding",
        "pdb_prefix": "DLG4",
    },
]

In [4]:
def find_split_file(split_dir: Path, split: str):
    candidates = [
        split_dir / f"{split}.tsv",
        split_dir / f"{split}.csv",
        split_dir / f"{split}_data.tsv",
        split_dir / f"{split}_data.csv",
    ]

    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(
        f"Could not find {split} file in {split_dir}. Tried:\n"
        + "\n".join(str(p) for p in candidates)
    )


def read_table(path: Path):
    if path.suffix.lower() == ".tsv":
        return pd.read_csv(path, sep="\t")
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")


def find_column(df: pd.DataFrame, candidates, required_name: str):
    cols = list(df.columns)
    lower_map = {c.lower(): c for c in cols}

    for c in candidates:
        if c in cols:
            return c

    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]

    raise ValueError(
        f"Could not find required column '{required_name}'. "
        f"Tried {candidates}. Available columns: {cols}"
    )

In [ ]:
VARIANT_COL_CANDIDATES = ["variant", "mutant", "mutation", "mutations", "sample_id", "name"]
SCORE_COL_CANDIDATES = ["score", "DMS_score", "target", "label", "fitness", "measurement"]


def make_variant_id(x: str):
    x = str(x).strip()
    x = x.replace(" ", "")
    x = x.replace("/", "-")
    x = x.replace(":", "-")
    x = x.replace(";", "-")

    if x.endswith(".pdb"):
        x = x[:-4]
    if x.endswith(".npz"):
        x = x[:-4]

    return x


def make_foldvision_protein_id(variant: str, pdb_prefix: str):
    variant = make_variant_id(variant)
    return f"{pdb_prefix}_{variant}.npz"


def load_split_as_foldvision_df(split_dir: Path, split: str, pdb_prefix: str):
    path = find_split_file(split_dir, split)
    raw = read_table(path)

    variant_col = find_column(raw, VARIANT_COL_CANDIDATES, "variant")
    score_col = find_column(raw, SCORE_COL_CANDIDATES, "score")

    out = raw[[variant_col, score_col]].copy()
    out.columns = ["variant", "label"]

    out["variant"] = out["variant"].map(make_variant_id)
    out["Protein ID"] = out["variant"].map(lambda x: make_foldvision_protein_id(x, pdb_prefix))
    out["label"] = pd.to_numeric(out["label"], errors="coerce")
    out["split"] = split

    before = len(out)
    out = out.dropna(subset=["variant", "Protein ID", "label"]).copy()
    after = len(out)

    if before != after:
        print(f"[WARN] Removed {before - after} invalid rows from {split}.")

    return out[["Protein ID", "label", "variant", "split"]]

In [6]:
summary_rows = []

for cfg in DATASETS:
    dataset = cfg["dataset"]
    split_dir = cfg["split_dir"]
    hpc_fv_name = cfg["hpc_fv_name"]

    print("\n" + "=" * 100)
    print(dataset)
    print("Split dir:", split_dir)

    local_dataset_dir = LOCAL_OUT_BASE / hpc_fv_name
    local_dataset_dir.mkdir(parents=True, exist_ok=True)

    split_dfs = {}

    for split in ["train", "val", "test"]:
        df_split = load_split_as_foldvision_df(
            split_dir=split_dir,
            split=split,
            pdb_prefix=cfg["pdb_prefix"],
        )
        split_dfs[split] = df_split

        out_csv = local_dataset_dir / f"{hpc_fv_name}_{split}.csv"
        df_split[["Protein ID", "label"]].to_csv(out_csv, index=False)

        print(f"{split}: {len(df_split)} -> {out_csv.name}")

    full_df = pd.concat(split_dfs.values(), ignore_index=True)
    full_df.to_csv(local_dataset_dir / f"{hpc_fv_name}_full_foldvision_table.tsv", sep="\t", index=False)

    # Basic checks
    train_set = set(split_dfs["train"]["Protein ID"])
    val_set = set(split_dfs["val"]["Protein ID"])
    test_set = set(split_dfs["test"]["Protein ID"])

    assert len(train_set & val_set) == 0, f"{dataset}: train/val overlap"
    assert len(train_set & test_set) == 0, f"{dataset}: train/test overlap"
    assert len(val_set & test_set) == 0, f"{dataset}: val/test overlap"

    assert full_df["Protein ID"].nunique() == len(full_df), f"{dataset}: duplicated Protein ID"

    summary_rows.append({
        "dataset": dataset,
        "hpc_fv_name": hpc_fv_name,
        "local_dataset_dir": str(local_dataset_dir),
        "hpc_data_dir": str(HPC_FOLDVISION_DATA_BASE / hpc_fv_name),
        "hpc_processed_dir": str(HPC_FOLDVISION_DATA_BASE / hpc_fv_name / "processed"),
        "hpc_pdb_dir": str(cfg["hpc_pdb_dir"]),
        "train": len(split_dfs["train"]),
        "val": len(split_dfs["val"]),
        "test": len(split_dfs["test"]),
        "total": len(full_df),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(LOCAL_OUT_BASE / "foldvision_preprocess_summary.tsv", sep="\t", index=False)

display(summary_df)


UBE4B
Split dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/UBE4B
train: 752 -> UBE4B_train.csv
val: 94 -> UBE4B_val.csv
test: 94 -> UBE4B_test.csv

GRB2
Split dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/GRB2
train: 827 -> GRB2_train.csv
val: 103 -> GRB2_val.csv
test: 104 -> GRB2_test.csv

PTEN_activity
Split dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/PTEN_activity
train: 5251 -> PTEN_activity_train.csv
val: 656 -> PTEN_activity_val.csv
test: 657 -> PTEN_activity_test.csv

PTEN_abundance
Split dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/PTEN_abundance
train: 3509 -> PTEN_abundance_train.csv
val: 439 -> PTEN_abundance_val.csv
test: 439 -> PTEN_abundance_test.csv

Alpha-Amylase_expression
Split dir: /Users/kevinbauer/Desktop/Masterarbeit/Supervised Phase/data/Final data/Alpha_Amylase_expression
train: 6060 -> Alpha-Amylase_expression_train.csv
val: 757 -> Alph

,dataset,hpc_fv_name,local_dataset_dir,hpc_data_dir,hpc_processed_dir,hpc_pdb_dir,train,val,test,total
0,UBE4B,UBE4B,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,752,94,94,940
1,GRB2,GRB2,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,827,103,104,1034
2,PTEN_activity,PTEN_activity,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,5251,656,657,6564
3,PTEN_abundance,PTEN_abundance,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,3509,439,439,4387
4,Alpha-Amylase_expression,Alpha-Amylase_expression,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,6060,757,758,7575
5,Alpha-Amylase_thermostability,Alpha-Amylase_thermostability,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,6059,757,758,7574
6,Alpha-Amylase_specific_activity,Alpha-Amylase_specific_activity,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,6060,757,758,7575
7,DLG4_abundance,DLG4_abundance,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,1024,128,128,1280
8,DLG4_binding,DLG4_binding,/Users/kevinbauer/Desktop/Masterarbeit/Supervi...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,/gpfs/project/projects/CompCellBio/PETaseTourn...,1152,144,145,1441


In [7]:
for cfg in DATASETS:
    hpc_fv_name = cfg["hpc_fv_name"]
    local_dataset_dir = LOCAL_OUT_BASE / hpc_fv_name

    print("\n" + "=" * 100)
    print(hpc_fv_name)
    print("=" * 100)

    for p in sorted(local_dataset_dir.glob("*")):
        print(p.name)


UBE4B
UBE4B_full_foldvision_table.tsv
UBE4B_test.csv
UBE4B_train.csv
UBE4B_val.csv

GRB2
GRB2_full_foldvision_table.tsv
GRB2_test.csv
GRB2_train.csv
GRB2_val.csv

PTEN_activity
PTEN_activity_full_foldvision_table.tsv
PTEN_activity_test.csv
PTEN_activity_train.csv
PTEN_activity_val.csv

PTEN_abundance
PTEN_abundance_full_foldvision_table.tsv
PTEN_abundance_test.csv
PTEN_abundance_train.csv
PTEN_abundance_val.csv

Alpha-Amylase_expression
Alpha-Amylase_expression_full_foldvision_table.tsv
Alpha-Amylase_expression_test.csv
Alpha-Amylase_expression_train.csv
Alpha-Amylase_expression_val.csv

Alpha-Amylase_thermostability
Alpha-Amylase_thermostability_full_foldvision_table.tsv
Alpha-Amylase_thermostability_test.csv
Alpha-Amylase_thermostability_train.csv
Alpha-Amylase_thermostability_val.csv

Alpha-Amylase_specific_activity
Alpha-Amylase_specific_activity_full_foldvision_table.tsv
Alpha-Amylase_specific_activity_test.csv
Alpha-Amylase_specific_activity_train.csv
Alpha-Amylase_specific_acti